<a href="https://colab.research.google.com/github/ArturBDev/ArturBDev/blob/main/tcc_gaussian_splatting_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Configurar variáveis de ambiente e diretório base


In [ ]:
import os

os.environ['CUDA_HOME'] = '/usr/local/cuda'
%cd /content

/content


# 2. Limpeza profunda para evitar que o Colab use cache quebrado


In [ ]:
!rm -rf gaussian-splatting tandt_db.zip tandt resultado_3dgs.zip
!pip uninstall -y diff-gaussian-rasterization simple-knn

# 3. Clonar o repositório oficial (INRIA)


In [ ]:
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting

Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 23.99 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (17

# 4. Instalar as dependências base do PyTorch


In [ ]:
!pip install -q plyfile ninja setuptools wheel
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.5 MB/s eta 0:00:00


# 5. Compilar os motores C++/CUDA do zero (sem a flag que causou erro)


In [ ]:
%cd /content/gaussian-splatting

!pip install --no-cache-dir ./submodules/diff-gaussian-rasterization/

!sed -i '1s/^/#include <float.h>\n/' ./submodules/simple-knn/simple_knn.cu
!pip install --no-cache-dir ./submodules/simple-knn/


/content/gaussian-splatting
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3812384 sha256=a715be766513b75afe727abec3fd3a5308ad78898f011aa12fdde350c7349f9f
  Stored in directory: /tmp/pip-ephem-wheel-cache-ji6otmsy/wheels/01/e0/e8/f40a1cd6a1d5760cbd3036081bdad5b36c41fc11c786d4a404
Successfully built diff_gaussian_rasterization
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3552483 sha256=9dd523421c484515fbf1771a4af800602312fceb08cd7da1b9a9bed3eb65e8ce
  Stored in directory: /tmp/pip-ephem-wheel-cache-6b5cx0uj/wheels/0a/f2/1b/255828ebad94ea248378281b7926639d83ce4f394f0052800d
Successfully built simple_knn


# 6. Baixar e extrair as imagens e as câmeras (dataset)


In [ ]:
%cd /content
!wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip -o -q tandt_db.zip

/content


# 7. Rodar o treinamento do 3DGS!


In [ ]:
%cd /content/gaussian-splatting
# Adicionamos a flag "-m /content/meu_modelo" para salvar em uma pasta de nome conhecido
!python train.py -s /content/tandt/train -m /content/meu_modelo --data_device cpu --optimizer_type default --antialiasing --iterations 7000

/content/gaussian-splatting
2026-05-24 20:09:28.917608: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/meu_modelo
Output folder: /content/meu_modelo [24/05 20:09:33]
Reading camera 301/301 [24/05 20:09:35]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [24/05 20:09:35]
Loading Training Cameras [24/05 20:09:36]
Loading Test Cameras [24/05 20:09:41]
Number of points at initialisation :  182686 [24/05 20:09:41]
Training progress: 100% 7000/7000 [08:58<00:00, 12.99it/s, Loss=0.0830419, Depth Loss=0.0000000]

[ITER 7000] Evaluating train: L1 0.06580297052860261 PSNR 20.136465072631836 [24/05 20:18:51]

[ITER 7000] Saving Gaussians [24/05 20:18:51]

Training complete. [24/05 20:18:58]


# 8. Renderizar as imagens para visualização no Colab


In [ ]:
# Isso vai ler o modelo recém-treinado e gerar imagens nas posições originais da câmera
!python render.py -m /content/meu_modelo

Looking for config file in /content/meu_modelo/cfg_args
Config file found: /content/meu_modelo/cfg_args
Rendering /content/meu_modelo
Loading trained model at iteration 7000 [24/05 20:19:12]
Reading camera 301/301 [24/05 20:19:14]
Loading Training Cameras [24/05 20:19:14]
Loading Test Cameras [24/05 20:19:19]
Rendering progress: 100% 301/301 [02:15<00:00,  2.23it/s]
Rendering progress: 0it [00:00, ?it/s]


# 9. Preparar os resultados para download


In [ ]:
# Compacta as imagens renderizadas e a nuvem de pontos (.ply) em um único arquivo zip
%cd /content
!zip -q -r resultado_3dgs.zip /content/meu_modelo/renders/ /content/meu_modelo/point_cloud/iteration_7000/point_cloud.ply

/content
